# Day 28 — APIs (Consuming)

**How to use this notebook:**
- Read each exercise, fill in your solution (replace `pass`)
- Run the test cell right after to see **APPROVED** or **FAILED**
- Do NOT modify the test cells

**Install first (run in terminal):**
```
pip install requests
```

**Key concepts today:** HTTP methods (GET, POST, PUT, DELETE), status codes, JSON responses, `requests` library, headers, query params, response parsing

---
## Exercise 1 — API Response Parser

Real APIs return verbose, nested JSON — far more than you need. Your job as a developer is to parse it cleanly and validate what you extract.

Write `parse_user_response(response)` that:
1. Navigates to `response['data']['user']` — raise `KeyError` with message `'Missing user data'` if the path does not exist
2. Extracts only these fields: `id`, `username`, `email`, `role`, `is_active`
3. Validates:
   - `id` must be an `int` — raise `ValueError('id must be an integer')` if not
   - `email` must contain `'@'` — raise `ValueError('Invalid email')` if not
   - `role` must be one of `['admin', 'user', 'moderator']` — raise `ValueError('Invalid role')` if not
4. Returns a clean dict with just those 5 fields

This mirrors exactly what you write when consuming a real user API (GitHub, Stripe, etc.).

In [ ]:
def parse_user_response(response):
    pass

In [ ]:
# TEST CELL — do not modify
def _check(label, got, expected):
    if got == expected:
        print(f'  ✔ APPROVED — {label}')
    else:
        print(f'  ✘ FAILED   — {label}')
        print(f'             got:      {got!r}')
        print(f'             expected: {expected!r}')

valid_response = {
    'status': 'success',
    'timestamp': '2024-01-15T08:32:11Z',
    'request_id': 'abc123xyz',
    'data': {
        'user': {
            'id': 42,
            'username': 'alice_dev',
            'email': 'alice@example.com',
            'full_name': 'Alice Smith',
            'role': 'admin',
            'is_active': True,
            'created_at': '2023-06-01',
            'metadata': {'theme': 'dark', 'language': 'en'}
        }
    }
}

print('\n── Exercise 1: parse_user_response ──')
result = parse_user_response(valid_response)
_check('id',        result.get('id'),        42)
_check('username',  result.get('username'),  'alice_dev')
_check('email',     result.get('email'),     'alice@example.com')
_check('role',      result.get('role'),      'admin')
_check('is_active', result.get('is_active'), True)
_check('no extra fields', set(result.keys()), {'id','username','email','role','is_active'})

# Missing data
try:
    parse_user_response({'data': {}})
    _check('missing user raises', False, True)
except KeyError as e:
    _check('missing user raises KeyError', True, True)

# Invalid email
bad_email = valid_response.copy()
bad_email['data'] = {'user': {**valid_response['data']['user'], 'email': 'not-an-email'}}
try:
    parse_user_response(bad_email)
    _check('bad email raises', False, True)
except ValueError:
    _check('bad email raises ValueError', True, True)

# Invalid role
bad_role = valid_response.copy()
bad_role['data'] = {'user': {**valid_response['data']['user'], 'role': 'superuser'}}
try:
    parse_user_response(bad_role)
    _check('bad role raises', False, True)
except ValueError:
    _check('bad role raises ValueError', True, True)

---
## Exercise 2 — Simple API Client

Build a reusable `APIClient` class that wraps the `requests` library.

- `__init__(self, base_url, headers=None)` — stores the base URL and default headers
- `get(self, endpoint, params=None)` — makes a GET request to `base_url + endpoint`
- `post(self, endpoint, data=None)` — makes a POST request with JSON body

Both `get` and `post` should:
- Use `try/except` to catch request failures (Day 17)
- Return a dict: `{'success': bool, 'status_code': int, 'data': dict_or_None, 'error': str_or_None}`
- `success` is `True` only if status code is 2xx

> Tests use `https://httpbin.org` — a free public API that echoes back your requests. Internet required.

In [ ]:
import requests

class APIClient:
    def __init__(self, base_url, headers=None):
        pass

    def get(self, endpoint, params=None):
        pass

    def post(self, endpoint, data=None):
        pass

In [ ]:
# TEST CELL — internet required (uses httpbin.org)
def _check(label, got, expected):
    if got == expected:
        print(f'  ✔ APPROVED — {label}')
    else:
        print(f'  ✘ FAILED   — {label}')
        print(f'             got:      {got!r}')
        print(f'             expected: {expected!r}')

print('\n── Exercise 2: APIClient ──')
client = APIClient('https://httpbin.org', headers={'X-Custom': 'test'})

get_result = client.get('/get', params={'name': 'alice'})
if 'error' not in get_result or get_result['error'] is None:
    _check('GET success',      get_result.get('success'),     True)
    _check('GET status code',  get_result.get('status_code'), 200)
    _check('GET data is dict', isinstance(get_result.get('data'), dict), True)
else:
    print(f'  ⚠ Skipped GET (no internet): {get_result["error"]}')

post_result = client.post('/post', data={'key': 'value'})
if 'error' not in post_result or post_result['error'] is None:
    _check('POST success',      post_result.get('success'),     True)
    _check('POST status code',  post_result.get('status_code'), 200)
    _check('POST data is dict', isinstance(post_result.get('data'), dict), True)
else:
    print(f'  ⚠ Skipped POST (no internet): {post_result["error"]}')

bad_result = client.get('/status/404')
if bad_result['error'] is None:
    _check('404 success=False', bad_result.get('success'), False)
    _check('404 status code',   bad_result.get('status_code'), 404)
else:
    print(f'  ⚠ Skipped 404 test (no internet)')

---
## Mini-Project — Weather Dashboard

Use the `requests` library to call the [Open-Meteo API](https://open-meteo.com/) (free, no API key needed) and build a multi-function weather tool.

---

### `get_current_weather(lat, lon)`
- Call `https://api.open-meteo.com/v1/forecast` with params:
  `latitude`, `longitude`, `current_weather=true`
- Extract from `response['current_weather']`: `temperature`, `windspeed`, `is_day` (convert to `bool`)
- Return `{'temperature': float, 'windspeed': float, 'is_day': bool}`
- On any failure → return `{'error': str(e)}`

---

### `get_forecast(lat, lon, days=3)`
- Call the same endpoint with additional params: `daily=temperature_2m_max,temperature_2m_min`, `forecast_days=days`, `timezone=auto`
- Return a list of dicts, one per day:
  `[{'date': str, 'max_temp': float, 'min_temp': float}, ...]`
- On any failure → return `{'error': str(e)}`

---

### `compare_cities(cities)`
- `cities` is a list of dicts: `[{'name': str, 'lat': float, 'lon': float}, ...]`
- Call `get_current_weather()` for each city
- Return a list of dicts sorted by temperature descending:
  `[{'city': str, 'temperature': float, 'windspeed': float}, ...]`
- Skip cities where the request failed

> Istanbul: lat=41.01, lon=28.97 | London: lat=51.51, lon=-0.13 | Tokyo: lat=35.69, lon=139.69

In [ ]:
import requests

def get_current_weather(lat, lon):
    pass


def get_forecast(lat, lon, days=3):
    pass


def compare_cities(cities):
    pass

In [ ]:
# TEST CELL — internet required
def _check(label, got, expected):
    if got == expected:
        print(f'  ✔ APPROVED — {label}')
    else:
        print(f'  ✘ FAILED   — {label}')
        print(f'             got:      {got!r}')
        print(f'             expected: {expected!r}')

print('\n── Mini-Project: Weather Dashboard ──')

# Current weather
w = get_current_weather(41.01, 28.97)
if 'error' not in w:
    _check('has temperature',    'temperature' in w,              True)
    _check('has windspeed',      'windspeed'   in w,              True)
    _check('has is_day',         'is_day'      in w,              True)
    _check('is_day is bool',     isinstance(w.get('is_day'), bool), True)
    _check('temperature is float', isinstance(w.get('temperature'), float), True)
    print(f'  Istanbul: {w}')
else:
    print(f'  ⚠ Skipped current weather (no internet): {w["error"]}')

# Forecast
f = get_forecast(41.01, 28.97, days=3)
if isinstance(f, list):
    _check('forecast length',     len(f),                          3)
    _check('has date',            'date'     in f[0],             True)
    _check('has max_temp',        'max_temp' in f[0],             True)
    _check('has min_temp',        'min_temp' in f[0],             True)
    print(f'  3-day forecast: {f}')
else:
    print(f'  ⚠ Skipped forecast (no internet)')

# Compare cities
cities = [
    {'name': 'Istanbul', 'lat': 41.01, 'lon': 28.97},
    {'name': 'London',   'lat': 51.51, 'lon': -0.13},
    {'name': 'Tokyo',    'lat': 35.69, 'lon': 139.69},
]
comparison = compare_cities(cities)
if isinstance(comparison, list) and len(comparison) > 0:
    _check('compare returns list',  isinstance(comparison, list),    True)
    _check('compare has 3 cities',  len(comparison),                 3)
    _check('compare has city key',  'city' in comparison[0],         True)
    _check('sorted by temp desc',
           comparison[0]['temperature'] >= comparison[-1]['temperature'], True)
    print(f'  City comparison: {comparison}')
else:
    print(f'  ⚠ Skipped comparison (no internet)')